In [ ]:
#Performing operations
using QuantumToolbox
using Random

v1 = basis(2,0)
v2 = basis(2,1)

sigmax() * v1
sigmay() * v2
sigmaz() * v1

H = Qobj([1/sqrt(2) 1/ sqrt(2); 1/sqrt(2) -1/sqrt(2)])
S = Qobj([1 0; 0  im])
T = Qobj([1 0 ; 0 exp(im*(pi/4))])

#=
function iterate(vector, list)
    for i in list
        vector=Qobj(i) * vector
    end
    return vector
end
iterate(basis(2,0), [[1 1 ; 1 -1 ], [1 0 ; 0 -1]])
#=







In [ ]:
#Measurement
using Plots
using QuantumToolbox

H = Qobj([1/sqrt(2) 1/ sqrt(2); 1/sqrt(2) -1/sqrt(2)])

function measure(vector)
    number = length(vector)
    prob_zero = abs(vector[1])^2
    

    if rand() < prob_zero
        measured = 0

    else
        measured = 1

    end

    return measured

end

function iterate(vector, trials)
    count_0 = 0
    count_1 = 0

    for i in 1:trials
        result = measure(vector)

        if result == 0
            count_0 += 1
        else 
            count_1 += 1

        end
    end

    bar(["State 0", "State 1"], [count_0, count_1],
    title= "Measurement Distribton",
    ylabel = "# of Occurences",
    color = [:blue, :pink])
end

qstate = H * basis(2,0)
iterate(qstate, 2000)



In [ ]:
#Vectors on a bloch sphere

using GLMakie

b = Bloch()
fig, _ = render(b)

point = [ 1/sqrt(3), 1/sqrt(3), 1 / sqrt(3)]
add_points!(b,point)
fig, _ = render(b)
fig

state1 = basis(2,0)
state2 = sigmax() * state1
state3 = H * state1
add_states!(b, [state1, state2, state3])



### Generate_Coordinates 
Loops over gates applies the physics
takes your starting state and gagtes and fils in the in between path for each operation(rotation)

____ MATH

d/dt |psi(t)> = -iH|psi(t)> -- Schrodinger Equations
|psi(t)> = e^(iHt)|psi(0)> Time Evolution Operator 

Use log to find the Hamilation that is used to drive the rotation

Plug the hamiltonian back inside and pslltit time into 40(or any) small steps calculates a fractional gate matrix(fraction of the whole gate) for every frame


### Get XYZ
Translates quantum vector in real 3d coordinates using expectation values extracts where the vector is pointing on a globe

### Animate
Feeds timeline into movie


In [ ]:
#Animation of the vectors 
using LinearAlgebra
using QuantumToolbox
using GLMakie

H_gate = Qobj((Array(sigmax().data)+ Array(sigmax().data)) / sqrt(2), type = Operator())
S_gate = Qobj([1.0+0.0im 0.0+0.0im; 0.0+0.0im 0.0+1.0im], type = Operator())

function generate_coordinates(initial, operations_list, steps=40, pause =5)
    function getxyz(state::QuantumObject)
        return [Real(expect(sigmax(), state)), Real(expect(sigmay(), state)), Real(expect(sigmaz(), state))]
    end

    timeline = Vector{Vector{Float64}}()
    current = initial
    push!(timeline, getxyz(current))

    for gate in operations_list
        H_matrix = im * log(Matrix(gate.data))

        for i in 1:steps
            fraction = i / steps
            U_fraction = exp(-im*H_matrix*fraction)
            interp_state = Qobj(U_fraction * current.data)
            push!(timeline, getxyz(interp_state))
        end

        current = gate * current
        push!(timeline, getxyz(current))
        final = getxyz(current)
        for i in 1:pause
            push!(timeline, final)
        end
    end

    return timeline
end

function animate(initial, operations_list)
    coords_timeline = generate_coordinates(initial, operations_list)
    b = Bloch()
    fig, lscene = render(b)

    first_frame = coords_timeline[1]
    line_points = Observable([Point3f(0,0,0), Point3f(first_frame[1], first_frame[2], first_frame[3])])

    lines!(lscene, line_points, color = :pink, linewidth = 10)
    record(fig, "quantum.mp4", coords_timeline; framerate = 30) do frame_coords
        line_points[] = [Point3f(0,0,0), Point3f(frame_coords[1], frame_coords[2], frame_coords[3])]
    end
    println("Done")
end


start = basis(2,0)
H = Qobj([1/sqrt(2) 1/ sqrt(2); 1/sqrt(2) -1/sqrt(2)])
operations = [H, S_gate]
animate(start, operations)


Done


In [ ]:
#Plot on Bloch Sphere using DiffEQ : Quantum TIime
#p = -i[H,t] = time evolution operator where H is the Hamiltonian(XYZ) and t is time
#p = 0.5[I + xX + yY + zZ] 
# for hamiltonnian Y x = 2z y=0 and z = 2x 
#  in terms of linblad MESOLVE for Hamiltonian Y  interference for x = 0 interfeence for y = -0.2y and z = -0.2z
#p = density matrix 
using DifferentialEquations
using QuantumToolbox
using GLMakie

import DifferentialEquations as DE, BenchmarkTools as BT



function lorenz!(du, u, p, t)
    du[1] = 2*u[3]
    du[2] = 0.0
    du[3] = 2*u[1]
end


function y_interference(u)
    u[1] = 0.0
    u[2] = -0.2 * u[2]
    u[3] = -0.2 * u[3]
end

u0 =[1.0; 0.0; 0.0]
tspan = (0.0, 20.0)
prob = DE.ODEProblem(lorenz!, u0, tspan)
sol = BT.@btime DE.solve(prob, DE.Tsit5());

b = Bloch()
fig, lscene = render(b)
coords_timeline = [sol(t) for t in sol.t]
first_frame = coords_timeline[1]
line_points = Observable([Point3f(0,0,0), Point3f(first_frame[1], first_frame[2], first_frame[3])])

lines!(lscene, line_points, color = :pink, linewidth = 10)
record(fig, "quantum.mp4", coords_timeline; framerate = 30) do frame_coords
    line_points[] = [Point3f(0,0,0), Point3f(frame_coords[1], frame_coords[2], frame_coords[3])]
end
println("Done")

  12.250 μs (842 allocations: 36.43 KiB)
Done
